# Baseline

Модель — топ популярных треков с экспоненциальным затуханием. Это обобщение простого топа по популярности, чтобы учесть изменчивость в предпочтениях за более короткие временные промежутки.

В статье [Yambda](https://arxiv.org/abs/2505.22238) соответствует модели *DecayPop* (см. секцию 4.2):

$$pop_i = \sum_{u \in U} \mathbf{1} [ listen(u, i) > 50\% ] \cdot \gamma^{t - T_{\max}}$$

- $t$ — время прослушивания;
- $T_{\max}$ — время последнего события в датасете, т.к. мы хотим топ популярных треков в конце датасета, чтобы прогнозировать будущие события;
- $\gamma$ — коэффициент дисконтирования.

#### 1. Скачиваем файл с прослушиваниями

Берём самое маленькое подмножество Yambda-50M

In [2]:
%%bash
FILENAME="listens.parquet"
URL="https://huggingface.co/datasets/yandex/yambda/resolve/main/flat/50m/${FILENAME}"
if [ -f ${FILENAME} ]; then
    echo "${FILENAME} already exists"
else
    wget -nv ${URL}
fi

listens.parquet already exists


In [3]:
import polars as pl
import numpy as np

#### 2. Читаем датасет

И сразу фильтруем, оставляя только позитивные прослушивания (>50%)

In [4]:
df = pl.read_parquet("listens.parquet").filter(pl.col("played_ratio_pct") > 50)

In [5]:
df.head()

uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
u32,u32,u32,u8,u16,u32
100,39420,8326270,0,100,170
100,39420,1441281,0,100,105
100,39625,286361,0,100,185
100,40110,732449,0,100,240
100,40380,7849270,0,100,205


In [7]:
max_ts = float(df["timestamp"].max())

####  3. Вычисляем популярность каждого трека

В данном случае используется $\gamma = 2^{\frac{1}{86400}}$, обеспечивая физический смысл дисконтирования как "период полураспада — 1 сутки". Это означает, что если прослушивание было $N$ дней назад, то оно учтётся с весом $1 / 2^N$

In [10]:
res = (
    df
    .group_by("item_id")
    .agg(
        (
            pl.lit(2).pow((pl.col("timestamp").cast(pl.Float64) - max_ts) / 86_400)
        )
        .sum()
        .alias("pop_1d")
    )
)

In [11]:
res.head()

item_id,pop_1d
u32,f64
7741731,5.8941e-68
9220561,0.158036
7127475,0.022868
9163037,0.000004
4943436,1.9770e-28


In [12]:
items_top_100 = res.top_k(100, by="pop_1d")["item_id"].to_list()

####  4. Читаем `uid`'ы тестовых пользователей

Файл `users.csv` можно скачать из описания задачи.

Т.к. это все пользователи из подмножества Yambda-50M, то их список можно получить как `range(100, 1_000_001, 100)`

In [14]:
test_uids = pl.read_csv("users.csv")["uid"].to_list()

#### 5. Сохраням решение в `test.csv`

Для каждого из тестовых `uid`'ов записываем одно и то же предсказание — топ-100 треков по популярности

In [ ]:
pl.from_dicts(
    [
        {"uid": i, "item_ids": " ".join(map(str, items_top_100))}
        for i in test_uids
    ]
).write_csv("test.csv")

Это решение должно получить `score = 49.86`, что соответствует $\text{Recall@100} = 4.986\%$.

Несмотря на то, что это достаточно простой бейзлайн, даже он масштабируется по количеству "обучающих" данных — если взять сэмпл Yambda-500M для вычисления популярности, то такое решение получит `score = 50.12`.

## Общие рекомендации

- изучить состав датасета [Yambda](https://huggingface.co/datasets/yandex/yambda)
    - помимо прослушиваний есть и другие полезные сигналы вроде лайков и дизлайков
    - доступен код разных бейзлайнов в папке [benchmarks](https://huggingface.co/datasets/yandex/yambda/tree/main/benchmarks)
- отлаживать пайплайны обучения на самом маленьком сэмпле, только потом переходить на полномасштабные обучения
- подбирать гиперпараметры моделей через temporal split выборки
- помнить о том, что зачастую ансамблирование работает лучше одного сильного бейзлайна
    - [nvidia blueprint](https://medium.com/nvidia-merlin/recommender-systems-not-just-recommender-models-485c161c755e): 4-Stage Recommender Systems
    - многие актуальные подходы совмещают в себе стадию кандидатогенерации и преранк:
        - GRank: https://arxiv.org/abs/2510.15299v1
        - SilverTorch: https://arxiv.org/abs/2511.14881v1